# APIG - M3 fine-tuning on Kaggle

**Before running**
1. *Settings* -> Accelerator: **GPU T4 x2** (or P100); Internet: **On**.
2. *Add Input* -> your **private** dataset that contains `apig_v1.parquet` (never make it public: licences).
3. Code comes from GitHub (`BRANCH` below). If the branch is not pushed yet, upload the repo's `src/` folder
   as another private dataset instead; the cell below finds it automatically.
4. *Save Version* -> **Save & Run All (Commit)** runs in the background; download `results.zip` from *Output*.

Runs already finished in `/kaggle/working/results` are skipped, so a re-run continues where it stopped.

In [ ]:
BRANCH = "main"          # branch that contains src/models/
SPLITS = "all"           # e.g. "split_random loso" for a subset
SEEDS = "0 1 2"
EXTRA = ""               # e.g. "--save-model" (keeps split_random weights for IG / demo), or "--smoke" to test

In [ ]:
import glob, os, shutil, subprocess, sys
REPO = "/kaggle/working/repo"
uploaded = glob.glob("/kaggle/input/*/src/models/m3_finetune.py") + glob.glob("/kaggle/input/*/*/src/models/m3_finetune.py")
if os.path.exists(REPO):
    print("repo already present")
elif uploaded:
    src_root = os.path.dirname(os.path.dirname(os.path.dirname(uploaded[0])))
    shutil.copytree(src_root, REPO)
    print("using uploaded code from", src_root)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    "https://github.com/jimmytrn154/Anti-Prompt-Injection-Generalization.git", REPO], check=True)
    print("cloned branch", BRANCH)
os.chdir(REPO)
DATA = [p for p in glob.glob("/kaggle/input/**/apig_v1*.parquet", recursive=True)]
print("data files:", DATA)
assert len(DATA) == 1, "attach exactly one apig_v1*.parquet dataset"
DATA = DATA[0]
if "FAKE" in DATA.upper():
    print("WARNING: FAKE labels - code test only, never report these results")

In [ ]:
import torch, transformers, sklearn, pandas
print(torch.__version__, transformers.__version__, sklearn.__version__, pandas.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable the GPU accelerator!")

In [ ]:
!python -m src.models.m3_finetune --data "{DATA}" --out /kaggle/working/results --splits {SPLITS} --seeds {SEEDS} {EXTRA}

In [ ]:
STEM = os.path.basename(DATA).rsplit(".", 1)[0]
!python -m src.models.summarize --results /kaggle/working/results/{STEM}
!cd /kaggle/working && zip -qr results.zip results && ls -lh results.zip